<div align="center">

# ☁️ aws-analyzer — every command, one notebook

**Readable reports on your AWS account, right in Jupyter.** Every report says what it found, what it costs,
and what to run next. Nothing here changes anything in AWS: every command only *reads*.

`S3` · `DynamoDB` · `Bedrock Knowledge Bases` · `Bedrock chat` · `SageMaker` · `OpenSearch` · `Lambda` ·
`Step Functions` · `API Gateway` · `Athena` · `SQS / SNS / EventBridge` · `✦ the agent`

</div>

---

### 🚀 How to use this notebook

| Step | What to do |
| :-: | :-- |
| **1** | Run **[Install](#install)** once (skip it if `aws-analyzer` is already installed). |
| **2** | Fill in **[⚙️ Your settings](#settings)** — the names of *your* bucket, table, function… — and run it. |
| **3** | Run **[✅ Check the connection](#check)** to see which AWS account and region you're on. |
| **4** | Jump to any service below and run the cells you want, **in any order**. Each service's first cell sets it up. |

> 💡 **Tips**
> - Every cell is one command. Change its arguments freely. Lines starting with `# ↳` are variations: delete the `# ↳ `
>   to run one.
> - Lost? Every service has a `.help()` cell that lists all its commands; `.help("name")` explains one in full.
> - A missing permission or package never crashes a cell — the report says what's missing and how to fix it.
> - Prefer clicking to typing? Look for the **🖱️ Explore by clicking** cell in each service: it opens a window.

<a id="contents"></a>
## 📑 Contents

| | Service | Start with | Click-through window |
| :-: | :-- | :-- | :-- |
| 🪣 | [Amazon S3](#s3) — buckets, folders, files, cost | `s3.overview()` | `S3Explorer()` |
| 🗂️ | [Amazon DynamoDB](#dynamodb) — tables, items, schema | `dynamo.tables()` | — |
| 📚 | [Bedrock Knowledge Bases](#bedrock-kb) — files, syncs, search, answers | `kb.kbs()` | `kb.explore()` |
| 💬 | [Bedrock knowledge base chat](#bedrock-chat) — a chat window, test questions | `kchat.app()` | `kchat.app()` |
| 🧪 | [Amazon SageMaker](#sagemaker) — this notebook, environments, jobs, endpoints | `sagemaker.instance()` | — |
| 🔎 | [Amazon OpenSearch](#opensearch) — vector indexes | `opensearch.overview()` | `opensearch.explore()` |
| λ | [AWS Lambda](#lambda) — functions, **logs**, errors, code | `lambdas.functions()` | `lambdas.explore()` |
| 🔀 | [AWS Step Functions](#stepfunctions) — state machines, executions | `sfn.overview()` | `sfn.explore()` |
| 🚪 | [Amazon API Gateway](#apigateway) — APIs, routes, calls | `apigw.apis()` | `apigw.explore()` |
| 🏛️ | [Amazon Athena & Glue](#athena) — queries, cost, catalog | `athena.overview()` | `athena.explore()` |
| 📨 | [SQS, SNS & EventBridge](#messaging) — where did my message go? | `msg.overview()` | `msg.explore()` |
| ✦ | [The agent](#agent) — ask about your account in plain English | `agent.setup()` | `agent.chat()` |
| 🧹 | [Housekeeping](#housekeeping) — free the disk | `local_files()` | — |

<details>
<summary><b>🧭 Quick recipes — "I want to…"</b> (click to open)</summary>

| I want to… | Go to |
| :-- | :-- |
| read a Lambda function's logs, run by run | [λ Lambda → Logs](#lambda-logs) |
| find out why a Lambda failed last night | [λ Lambda → When something goes wrong](#lambda-logs) |
| click through my S3 buckets like a file explorer | [🪣 S3 → Explore by clicking](#s3-explore) |
| peek inside a CSV / Parquet / PDF in S3 | [🪣 S3 → Open a file](#s3-open) |
| find what's costing me money in S3 | [🪣 S3 → Cut cost](#s3-cost) |
| see why a Step Functions execution failed | [🔀 Step Functions → Executions](#sfn-executions) |
| see which API calls fail and the Lambda run behind each | [🚪 API Gateway → Calls](#apigw-calls) |
| find my most expensive Athena queries | [🏛️ Athena → Past queries](#athena-queries) |
| find messages stuck in a dead-letter queue | [📨 Messaging](#messaging) |
| see what's running (and billing) in SageMaker | [🧪 SageMaker → Your account](#sagemaker) |
| ask a question in plain English | [✦ The agent](#agent) |

</details>

---
<a id="install"></a>
# 📦 Install

Run once per environment. Only `boto3` is required (SageMaker already has it); `[all]` adds pandas, pyarrow, the PDF and
Excel readers, progress bars and widgets, so every command works. Restart the kernel after installing.

In [ ]:
%pip install --quiet "aws-analyzer[all]"
# %pip install --quiet aws-analyzer                 # the smallest install: boto3 + the agent's packages
# %pip install --quiet "aws-analyzer[sandbox]"      # also Deno, for the agent's Python sandbox (python="local")

<a id="settings"></a>
# ⚙️ Your settings

**Fill these in once.** Every cell below reads them, so you can run any command without editing it.
Leave a name as it is if you don't use that service. `REGION = None` and `PROFILE = None` use the notebook's own
region and IAM role, which is what you want on SageMaker.

> ✍️ Names are forgiving: a bucket can be `s3://bucket/prefix` or `bucket/prefix`; a Lambda function can be its name,
> `"name:alias"`, its ARN or a console link; times can be `"24h"`, `"7d"` or `"2024-05-01"`; sizes `"10MB"`.

In [ ]:
# ── Where ────────────────────────────────────────────────────────────────────────────
REGION = None           # e.g. "eu-west-1"; None = the notebook's region
PROFILE = None          # a profile from ~/.aws/config; None = the notebook's IAM role
TIMEZONE = None         # e.g. "Asia/Kolkata", "America/New_York", "local"; None = UTC
THEME = None            # "console" (default), "soft", "editorial", "brand" or "classic"

# ── 🪣 S3 ────────────────────────────────────────────────────────────────────────────
BUCKET = "my-bucket"
PREFIX = "data/"                        # a folder inside the bucket
FILE = "data/part-0.parquet"            # a file inside the bucket (csv, parquet, json, pdf, docx, ...)
DOCUMENT = "docs/model-card.docx"      # a PDF, Word or PowerPoint file inside the bucket

# ── 🗂️ DynamoDB ──────────────────────────────────────────────────────────────────────
TABLE = "orders"
PARTITION = "USER#42"                   # a partition key value
SORT = "ORDER#0017"                     # a sort key value (if the table has one)
ATTRIBUTE = "status"                    # an attribute to count values of

# ── 📚 Bedrock knowledge bases ───────────────────────────────────────────────────────
KNOWLEDGE_BASE = "support-docs"         # a name or an ID
KB_FILE = "refund-policy.pdf"           # a file in it (a name or its s3:// path)
QUESTION = "How long do refunds take?"

# ── 🔎 OpenSearch ────────────────────────────────────────────────────────────────────
OPENSEARCH = "vectors-prod"             # a domain or a Serverless collection
OS_INDEX = "docs"                       # a vector index in it

# ── λ Lambda ────────────────────────────────────────────────────────────────────────
FUNCTION = "orders-etl"

# ── 🔀 Step Functions ────────────────────────────────────────────────────────────────
STATE_MACHINE = "order-pipeline"
EXECUTION = "order-pipeline/run-0001"   # "state-machine/execution-name", or an execution ARN

# ── 🚪 API Gateway ───────────────────────────────────────────────────────────────────
API = "orders-api"

# ── 🏛️ Athena ────────────────────────────────────────────────────────────────────────
ATHENA_TABLE = "sales.orders"           # "database.table"
QUERY_ID = "00000000-0000-0000-0000-000000000000"   # a past query's ID (from athena.queries())

# ── 📨 SQS / SNS / EventBridge ───────────────────────────────────────────────────────
QUEUE = "email-jobs"
TOPIC = "order-events"
RULE = "order-placed"

# ── 🧪 SageMaker ─────────────────────────────────────────────────────────────────────
TRAINING_JOB = "churn-xgb-2024-05-01"
PIPELINE = "churn-training"

# ── Made from the names above (no need to edit) ─────────────────────────────────────
FOLDER = f"s3://{BUCKET}/{PREFIX}"
FILE_URI = f"s3://{BUCKET}/{FILE}"
DOCUMENT_URI = f"s3://{BUCKET}/{DOCUMENT}"
WHERE = {"region": REGION, "profile": PROFILE}
print("✅ Settings saved. Next: run the connection check below.")

<a id="check"></a>
## ✅ Check the connection

Says which AWS account, identity and region the commands will read. If this fails, nothing else will work yet: it tells
you why.

In [ ]:
import boto3
from botocore.exceptions import BotoCoreError, ClientError

from aws_analyzer import __version__, set_theme, set_timezone

try:
    session = boto3.Session(profile_name=PROFILE, region_name=REGION)
    me = session.client("sts").get_caller_identity()
    print(f"✅ aws-analyzer {__version__}")
    print(f"   Account : {me['Account']}")
    print(f"   Identity: {me['Arn']}")
    print(f"   Region  : {session.region_name or '(not set: pass REGION above)'}")
except (BotoCoreError, ClientError) as error:
    print(f"❌ Can't reach AWS: {error}")
    print("   On SageMaker this uses the notebook's IAM role; elsewhere run `aws configure` or set PROFILE above.")

if TIMEZONE:
    print("🕒 Times shown in", set_timezone(TIMEZONE))
if THEME:
    print("🎨 Reports drawn in the", set_theme(THEME), "theme")

---
<a id="s3"></a>
# 🪣 Amazon S3

**Module:** `aws_analyzer.s3` &nbsp;·&nbsp; 📖 [Full guide](https://utkarsh5026.github.io/aws-analyzer/s3.html) &nbsp;·&nbsp; ⬆ [Back to contents](#contents)

| If you want to know… | Run |
| :-- | :-- |
| What buckets do I have, what do they cost, are any public? | `s3.overview()` |
| What's in this folder and what does it cost? | `s3.summary(FOLDER)` |
| What's inside this file? | `s3.preview(FILE_URI)` |
| How much would a lifecycle rule save? | `s3.what_if(FOLDER, move_after=30, to='STANDARD_IA')` |
| Can I get back a file I deleted? | `s3.deleted(FOLDER, deleted_after='7d')` |

**Uses from the settings cell:** `BUCKET`, `PREFIX`, `FILE`, `DOCUMENT`

In [ ]:
from aws_analyzer import S3Analyzer, S3View

s3 = S3View(S3Analyzer(**WHERE))
s3.help()                     # every S3 command, grouped by task; s3.help("summary") shows one in full

<a id="s3-explore"></a>
## 🖱️ Explore by clicking

A file explorer for S3: folders on the left, click a file to see inside it on the right (tables, PDFs, Word files,
archives…). Search, filter by type, include subfolders, tick files and download them as one `.zip`.

In [ ]:
from aws_analyzer import S3Explorer

x = S3Explorer(FOLDER, core=s3.core)    # or S3Explorer() to start from the list of buckets
x
# ↳ x.open("s3://other-bucket/logs/")     # go somewhere from code; x.back(), x.up(), x.refresh()

## 🪣 Buckets

In [ ]:
# Every bucket in one table: size, estimated monthly cost, versioning, encryption, public access and warnings
s3.overview()
# ↳ s3.overview("sagemaker-*")            # only buckets whose name matches

In [ ]:
# All buckets with their region, creation date and age (fast: no CloudWatch)
s3.buckets()

In [ ]:
# One bucket: settings, policy in plain English, risks, size and monthly cost
s3.bucket_info(BUCKET)

In [ ]:
# The bucket policy in plain English: who can do what, on which files, when
s3.policy(BUCKET)

## 📂 Explore a folder

In [ ]:
# Folders and files directly under a folder (one level, fast)
s3.ls(FOLDER)
# ↳ s3.ls(FOLDER, details=True)           # also what's inside each file: pages, rows, columns

In [ ]:
# A folder tree with object count and size at every level
s3.tree(FOLDER, depth=2)

In [ ]:
# The full dashboard for a folder: totals, monthly cost, findings, file types, storage classes, ages
s3.summary(FOLDER)

In [ ]:
# Search by name, extension, size, date or storage class
s3.find(FOLDER, pattern="*.csv", min_size="10MB", modified_after="7d")
# ↳ s3.find(FOLDER, extensions=["parquet", "json"])
# ↳ s3.find(FOLDER, regex=r"part-\d+\.parquet$")

In [ ]:
# What's inside each file: a PDF's pages, a Word file's words, a spreadsheet's sheets, a table's columns
s3.file_details(FOLDER, limit=50)

In [ ]:
# The biggest, newest and oldest files under a folder
s3.largest(FOLDER, 20)
# ↳ s3.newest(FOLDER, 20)
# ↳ s3.oldest(FOLDER, 20)

In [ ]:
# Compare two folders, e.g. to check a copy or a migration
s3.compare(FOLDER, f"s3://{BUCKET}/backup/{PREFIX}")

<a id="s3-cost"></a>
## 💰 Cut cost

In [ ]:
# Identical files: what the copies take and cost, and folders that hold only copies
s3.duplicates(FOLDER)

In [ ]:
# Preview a lifecycle rule before adding it: what it would move or delete today, and the money saved
s3.what_if(FOLDER, move_after=30, to="STANDARD_IA")
# ↳ s3.what_if(FOLDER, move_after={30: "STANDARD_IA", 90: "GLACIER"}, delete_after=365)

In [ ]:
# Incomplete multipart uploads: billed, but invisible in normal listings
s3.uploads(FOLDER)

## 🕘 Versions and deleted files

These need a bucket with versioning turned on.

In [ ]:
# Current vs older versions and delete markers: hidden storage in versioned buckets
s3.versions(FOLDER)

In [ ]:
# One file's version history
s3.history(FILE_URI)

In [ ]:
# Deleted files you can still bring back, most recent first, with the call that restores each
s3.deleted(FOLDER, deleted_after="7d")

<a id="s3-open"></a>
## 📄 Open a file

`preview` reads CSV, TSV, JSON, JSON Lines, Parquet, ORC, Feather, Avro, Excel, NumPy, archives, images, PDFs, Word,
PowerPoint and plain text.

In [ ]:
# Peek inside a file: a table's first rows and schema, a PDF's pages, an archive's contents ...
s3.preview(FILE_URI)

In [ ]:
# Every piece of metadata of one object
s3.head(FILE_URI)

In [ ]:
# A document as it reads: a Word file with its headings and pictures, a PDF's text laid out
s3.document(DOCUMENT_URI)
# ↳ s3.document(DOCUMENT_URI, pages=[1, 2, 3])   # only some pages of a PDF

In [ ]:
# A clickable download link that works without an AWS login, for an hour
s3.link(FILE_URI, expires=3600)

In [ ]:
# Load a file straight into pandas (needs pandas)
df = s3.core.read_df(FILE_URI)
df.head()

## ⬇ Download

Downloads go to `s3-downloads/` next to this notebook unless you give a path.

In [ ]:
# Download a file, or a whole folder; files already downloaded are skipped
s3.download(FILE_URI)
# ↳ s3.download(FOLDER)

In [ ]:
# Download as one .zip, after checking it fits (size, file count, disk, memory)
s3.download_zip(FOLDER, dry_run=True)       # dry_run=True only checks; remove it to download

In [ ]:
# What you've downloaded, how much disk it takes, and how to clean it up
s3.downloads()

In [ ]:
# Free the disk: delete what download() and download_zip() saved
s3.clean_downloads(older_than="7d", dry_run=True)   # dry_run=True only lists what would go

---
<a id="dynamodb"></a>
# 🗂️ Amazon DynamoDB

**Module:** `aws_analyzer.dynamodb` &nbsp;·&nbsp; 📖 [Full guide](https://utkarsh5026.github.io/aws-analyzer/dynamodb.html) &nbsp;·&nbsp; ⬆ [Back to contents](#contents)

| If you want to know… | Run |
| :-- | :-- |
| What tables do I have, how big, what do they cost? | `dynamo.tables()` |
| How do I query this table and its indexes? | `dynamo.table_info(TABLE)` |
| What do the items look like? | `dynamo.sample(TABLE) / dynamo.schema(TABLE)` |
| Is a partition hot? | `dynamo.value_counts(TABLE, 'pk')` |

**Uses from the settings cell:** `TABLE`, `PARTITION`, `SORT`, `ATTRIBUTE`

In [ ]:
from aws_analyzer import DynamoDBAnalyzer, DynamoDBView

dynamo = DynamoDBView(DynamoDBAnalyzer(**WHERE))
dynamo.help()

## 🗂️ Tables

In [ ]:
# Every table: key, items, size, billing mode, monthly cost and warnings
dynamo.tables()
# ↳ dynamo.tables("prod-*")

In [ ]:
# Keys, indexes and the query(...) call for each, capacity, usage, backups, cost and risks
dynamo.table_info(TABLE)

## 👀 Look at items

`where=` filters items: `{'status': 'failed', 'total': ('>', 100)}`. `more()` shows the next page.

In [ ]:
# About 20 items spread across the whole table: the best first look
dynamo.sample(TABLE)

In [ ]:
# Items from the start of the table, filtered
dynamo.scan(TABLE, where={"status": "failed"})

In [ ]:
# The next page of the last scan / query / sql
dynamo.more()

In [ ]:
# Items that share one partition key, in sort-key order
dynamo.query(TABLE, PARTITION)
# ↳ dynamo.query(TABLE, PARTITION, sort=("begins_with", "ORDER#"))
# ↳ dynamo.query(TABLE, "shipped", index="status-index")      # query a secondary index

In [ ]:
# One item by its key, with nested maps and lists expanded
dynamo.get(TABLE, PARTITION, SORT)

In [ ]:
# PartiQL (SQL-like); without the partition key in WHERE it scans the table
dynamo.sql(f'SELECT * FROM "{TABLE}" WHERE pk = ?', PARTITION)

## 🔬 Understand the data

In [ ]:
# Every attribute's types, fill rate, examples and range, and key patterns
dynamo.schema(TABLE)

In [ ]:
# How often each value of an attribute occurs (on the partition key: hot partitions)
dynamo.value_counts(TABLE, ATTRIBUTE)

In [ ]:
# The biggest items (the limit is 400 KB)
dynamo.largest(TABLE)

In [ ]:
# An exact count of items: reads (and pays for) every item, so mind big tables
dynamo.count(TABLE, where={"status": "failed"})

---
<a id="bedrock-kb"></a>
# 📚 Amazon Bedrock Knowledge Bases

**Module:** `aws_analyzer.bedrock_kb` &nbsp;·&nbsp; 📖 [Full guide](https://utkarsh5026.github.io/aws-analyzer/bedrock_kb.html) &nbsp;·&nbsp; ⬆ [Back to contents](#contents)

| If you want to know… | Run |
| :-- | :-- |
| What knowledge bases do I have, are they synced? | `kb.kbs()` |
| Which files failed to index, and why? | `kb.files(status='failed')` |
| Why doesn't my question find this file? | `kb.search_file(KB_FILE, QUESTION)` |
| What would it answer, with citations? | `kb.ask(QUESTION)` |

**Uses from the settings cell:** `KNOWLEDGE_BASE`, `KB_FILE`, `QUESTION`

In [ ]:
from aws_analyzer import BedrockKBAnalyzer, BedrockKBView

kb = BedrockKBView(BedrockKBAnalyzer(**WHERE), kb=KNOWLEDGE_BASE)   # kb= is the default for every command
kb.help()

## 🖱️ Explore by clicking

Every file next to Bedrock's record of it, how each was indexed, syncs, search and settings.

In [ ]:
kb.explore()
# ↳ kb.explore(file=KB_FILE)          # straight to one file

## 📚 Knowledge bases

In [ ]:
# Every knowledge base: status, vector store, embedding model, documents, last sync, idle cost, warnings
kb.kbs()

In [ ]:
# Switch the knowledge base later commands use
kb.use(KNOWLEDGE_BASE)

In [ ]:
# Settings in plain English, data sources, recent syncs, findings and cost
kb.kb_info()

## 📥 What's indexed

In [ ]:
# Every file next to Bedrock's record of it: indexed, failed (and why), changed, not synced yet
kb.files()
# ↳ kb.files(status="failed")
# ↳ kb.files(match="policy")

In [ ]:
# How one file was indexed, step by step: stored, parsed, cut into chunks (with each chunk's text)
kb.file(KB_FILE)

In [ ]:
# Whether a question finds a file, and where it ranks among the whole knowledge base
kb.search_file(KB_FILE, QUESTION)

In [ ]:
# Sync history: when, how long, counts, and why syncs failed
kb.syncs()

In [ ]:
# Documents by status, Bedrock's reason for those not indexed, and the sync command
kb.documents()

In [ ]:
# Files added or changed in S3 since the last sync
kb.unsynced()

In [ ]:
# Save the list of files as a .txt, with a link to download it
kb.save_file_list()

## 🔎 Search and answer

In [ ]:
# Ranked passages for a question, with scores, sources and highlighted words
kb.search(QUESTION, n=5)
# ↳ kb.search(QUESTION, search_type="HYBRID", where={"category": "billing"})

In [ ]:
# The full text of result #1 of the last search or ask
kb.chunk(1)

In [ ]:
# A link that opens source #1 in a new tab (a PDF at the passage's page)
kb.link(1)

In [ ]:
# An answer with [1][2] citations, grounded %, model, tokens, cost and time
kb.ask(QUESTION)
# ↳ kb.ask(QUESTION, model="sonnet")

In [ ]:
# A follow-up in the same conversation
kb.follow_up("And for international orders?")

In [ ]:
# Models you can use for ask(), and their price per 1M tokens
kb.models()

## 📏 Measure retrieval

In [ ]:
# One question under several search settings, side by side
kb.compare(QUESTION, n=(5, 10), search_types=("SEMANTIC", "HYBRID"))

In [ ]:
# Hit rate and MRR on test questions: (question, the file you expect it to find)
kb.evaluate([
    (QUESTION, KB_FILE),
    ("Can I change my delivery address?", "shipping-faq.md"),
])

---
<a id="bedrock-chat"></a>
# 💬 Bedrock knowledge base chat

**Module:** `aws_analyzer.bedrock_chat` &nbsp;·&nbsp; 📖 [Full guide](https://utkarsh5026.github.io/aws-analyzer/bedrock_chat.html) &nbsp;·&nbsp; ⬆ [Back to contents](#contents)

| If you want to know… | Run |
| :-- | :-- |
| Chat with my knowledge base and tweak every setting | `kchat.app()` |
| How does a list of test questions do? | `kchat.ask_all([...])` |
| Which settings answer best? | `kchat.sweep(n=[5, 10], ...)` |
| Give me this setup as code | `kchat.code()` |

**Uses from the settings cell:** `KNOWLEDGE_BASE`, `QUESTION`

In [ ]:
from aws_analyzer import BedrockChatAnalyzer, BedrockChatView

kchat = BedrockChatView(BedrockChatAnalyzer(**WHERE), kb=KNOWLEDGE_BASE, model="sonnet")
kchat.help()

## 💬 Chat

In [ ]:
# 🖱️ The chat window: pick the knowledge base and model, ask, see the request and response, run tests
kchat.app()

In [ ]:
# An answer as a report (follow-ups keep the conversation)
kchat.ask(QUESTION)

In [ ]:
# Only the search behind an answer: every passage retrieved, best first
kchat.retrieve(QUESTION)

In [ ]:
# The last answer in full: every cited passage, the exact request and response
kchat.last()

In [ ]:
# The whole conversation as a report that stays in the saved notebook
kchat.transcript()

In [ ]:
# Start a new conversation (settings stay)
kchat.new_chat()

## 🧪 Test a list of questions

In [ ]:
# Ask test questions, each on its own; "question | expected file" checks the source cited
kchat.ask_all([
    f"{QUESTION} | refund-policy.pdf",
    "Can I change my delivery address?",
])

In [ ]:
# Every combination of settings on the test questions, best setup first
kchat.sweep(n=[5, 10], search_type=["SEMANTIC", "HYBRID"])

In [ ]:
# A test run again (the last one by default)
kchat.results()

## 📈 Keep and compare test runs

In [ ]:
kchat.runs()                 # every test run so far
# ↳ kchat.compare_runs(1, 2)   # two runs side by side
# ↳ kchat.use_run(2)           # switch to run 2's setup

In [ ]:
# Save the runs to a file to keep after a restart, and read them back
kchat.save_runs()
# ↳ kchat.load_runs()

## ⚙️ Settings

In [ ]:
# What's sent with every question, in plain English
kchat.settings()

In [ ]:
# Change what's sent; unset() goes back to Bedrock's defaults
kchat.set(temperature=0.2, n=8)
# ↳ kchat.unset("temperature")

In [ ]:
# Every setting RetrieveAndGenerate takes
kchat.fields()
# ↳ kchat.fields("prompt")

In [ ]:
# The exact request your next question sends (nothing is sent)
kchat.request(QUESTION)

In [ ]:
# This setup as Python (boto3 only), JSON and an AWS CLI command, to copy
kchat.code()

## 📚 Knowledge base, files and model

In [ ]:
kchat.kbs()                  # knowledge bases you can chat with
# ↳ kchat.use(kb=KNOWLEDGE_BASE, model="haiku")

In [ ]:
kchat.files()                # the knowledge base's files, to point questions at
# ↳ kchat.use(files=["refund-policy.pdf"])
# ↳ kchat.save_file_list()

In [ ]:
kchat.models()               # models you can chat with, and their prices
# ↳ kchat.inference_profiles()

In [ ]:
# A one-line chat window, without making a View first
from aws_analyzer import chat

chat(KNOWLEDGE_BASE, model="sonnet", **WHERE)

---
<a id="sagemaker"></a>
# 🧪 Amazon SageMaker

**Module:** `aws_analyzer.sagemaker_env` &nbsp;·&nbsp; 📖 [Full guide](https://utkarsh5026.github.io/aws-analyzer/sagemaker_env.html) &nbsp;·&nbsp; ⬆ [Back to contents](#contents)

| If you want to know… | Run |
| :-- | :-- |
| What is this notebook costing me, does it stop when idle? | `sagemaker.instance()` |
| What's filling the disk? | `sagemaker.disk()` |
| What's running and billing right now? | `sagemaker.running()` |
| Why did my training job fail? | `sagemaker.training_job(TRAINING_JOB)` |
| Which endpoints does nobody call? | `sagemaker.endpoints()` |

**Uses from the settings cell:** `TRAINING_JOB`, `PIPELINE`

In [ ]:
from aws_analyzer import SageMakerAnalyzer, SageMakerView

sagemaker = SageMakerView(SageMakerAnalyzer(**WHERE))
sagemaker.help()

## 💻 This notebook

In [ ]:
# This notebook: instance type and cost, uptime, idle shutdown, CPU, memory, disk and GPU
sagemaker.instance()

In [ ]:
# What fills the disk, and the caches and trash that are safe to clear (with the command for each)
sagemaker.disk()

## 🐍 Python environments

In [ ]:
# The Python environments and Jupyter kernels on this machine
sagemaker.envs()

In [ ]:
# A new Python environment with its own Jupyter kernel, kept when the notebook stops
sagemaker.new_env("3.11", packages=["pandas", "scikit-learn"], dry_run=True)   # remove dry_run=True to make it

## ☁️ Your account

In [ ]:
# Everything SageMaker bills by the hour in this region: notebooks, Studio apps, endpoints, jobs
sagemaker.running()

In [ ]:
# Who opened, started or stopped notebooks in the last 30 days, and from where
sagemaker.access()

## 🏋️ Training jobs

In [ ]:
# The last 7 days of training jobs: how each ended, how long, the cost, and why failed ones failed
sagemaker.training_jobs()
# ↳ sagemaker.training_jobs(since="30d", status="Failed")

In [ ]:
# One training job: why it failed and what to change, the error in its log, its timeline
sagemaker.training_job(TRAINING_JOB)

## 🔌 Endpoints and 🔀 pipelines

In [ ]:
# Every endpoint with its monthly cost and calls; idle ones first
sagemaker.endpoints()

In [ ]:
# Pipelines and how their latest runs went
sagemaker.pipelines()

In [ ]:
# One pipeline run: the step that broke and why, drawn as a graph
sagemaker.pipeline_run(PIPELINE)

---
<a id="opensearch"></a>
# 🔎 Amazon OpenSearch (vector indexes)

**Module:** `aws_analyzer.opensearch` &nbsp;·&nbsp; 📖 [Full guide](https://utkarsh5026.github.io/aws-analyzer/opensearch.html) &nbsp;·&nbsp; ⬆ [Back to contents](#contents)

| If you want to know… | Run |
| :-- | :-- |
| What domains and collections do I have, what do they cost? | `opensearch.overview()` |
| Which indexes hold vectors and how much memory do they need? | `opensearch.indexes()` |
| Do documents miss vectors? | `opensearch.sample()` |
| What comes back for a question? | `opensearch.search('...')` |

**Uses from the settings cell:** `OPENSEARCH`, `OS_INDEX`

In [ ]:
from aws_analyzer import OpenSearchAnalyzer, OpenSearchView

opensearch = OpenSearchView(OpenSearchAnalyzer(**WHERE))
opensearch.use(f"{OPENSEARCH}/{OS_INDEX}")      # the domain and index later commands use
opensearch.help()

## 🖱️ Explore by clicking

In [ ]:
opensearch.explore()

## 🗂️ Domains, collections and indexes

In [ ]:
# Every domain and Serverless collection: what it runs on, monthly cost and warnings
opensearch.overview()

In [ ]:
# Every index: documents, size, shards, vector fields, documents missing a vector, memory needed
opensearch.indexes()

In [ ]:
# One index in plain English: vector fields, engine, similarity, and what a score means
opensearch.index_info()

## 🧭 Vectors

In [ ]:
# A few random documents, and a check of 200 documents' vectors
opensearch.sample()

In [ ]:
# The 10 documents nearest a question (embedded with Bedrock)
opensearch.search(QUESTION, k=10)
# ↳ opensearch.search(like="doc-id-123")      # documents like one you have

---
<a id="lambda"></a>
# λ AWS Lambda

**Module:** `aws_analyzer.lambda_functions` &nbsp;·&nbsp; 📖 [Full guide](https://utkarsh5026.github.io/aws-analyzer/lambda_functions.html) &nbsp;·&nbsp; ⬆ [Back to contents](#contents)

| If you want to know… | Run |
| :-- | :-- |
| What functions do I have, which are failing, what do they cost? | `lambdas.functions()` |
| Why did this function fail? | `lambdas.errors(FUNCTION)` |
| Show me its logs, run by run | `lambdas.logs(FUNCTION)` |
| Which runs were slow or cold? | `lambdas.runs(FUNCTION, where='duration > 5s')` |
| Does it need more memory? | `lambdas.performance(FUNCTION)` |
| How does its code work? | `lambdas.how_it_works(FUNCTION)` |

**Uses from the settings cell:** `FUNCTION`

In [ ]:
from aws_analyzer import LambdaAnalyzer, LambdaView

lambdas = LambdaView(LambdaAnalyzer(**WHERE))
lambdas.help()

## 🖱️ Explore by clicking

The **Lambda explorer**: every function, the ones that need attention first. Click one for what's wrong, its **logs
run by run** (failed runs in red, click a run for every line it logged), errors by cause, run times, code and
settings. The **Live** tab shows what's running now.

| Tab | What you see |
| :-- | :-- |
| **Functions** | every function: runtime, triggers, calls, error rate, cost, warnings (search, chips, sort) |
| **Overview** | the findings with what to do, how it's wired, 30 days of calls |
| **Logs** | a line per run; type to filter, **Enter** to search CloudWatch, or a query like `duration > 15s and cold` |
| **Errors / Performance** | failed runs grouped by cause; run times, memory, cold starts |
| **Code** | the package's files, the source in colour, **How it works**, **Download .zip** |
| **Live** | what's running right now, refreshed every few seconds |

In [ ]:
lambdas.explore()                          # every function in the region
# ↳ lambdas.explore(FUNCTION, tab="logs")    # straight to one function's logs
# ↳ lambdas.explore(tab="live")              # what's running now
# ↳ lambdas.explore(region="all")            # every region your account has turned on

In [ ]:
# Drive the window from code (open it with the cell above first)
x = lambdas.explorer
x.logs(search="KeyError", since="24h")     # the Logs tab, searched
# ↳ x.open("another-function")

## λ Functions

In [ ]:
# Every function: runtime (and when it loses support), memory, timeout, triggers, calls, errors, cost, warnings
lambdas.functions()
# ↳ lambdas.functions(regions="all")          # every region
# ↳ lambdas.functions("orders-*")             # only matching names

In [ ]:
# One function in plain English: what it runs, what triggers it, who may call it, and its last 30 days
lambdas.function_info(FUNCTION)

<a id="lambda-logs"></a>
## 🩺 When something goes wrong — errors and **logs**

| To… | Run |
| :-- | :-- |
| see errors grouped by cause (timeouts, out of memory, permissions, exceptions) | `lambdas.errors(FUNCTION)` |
| read the newest runs like a log viewer | `lambdas.logs(FUNCTION)` |
| search the logs for a word | `lambdas.logs(FUNCTION, search="KeyError", since="24h")` |
| read one run by its request ID | `lambdas.logs(FUNCTION, request_id="8f5ce35b-...")` |
| query runs by their fields | `lambdas.runs(FUNCTION, where="duration > 15s and cold")` |

Times take `"15m"`, `"24h"`, `"7d"` or `"2024-05-01 10:00"`. Searches take `AND`, `OR`, `NOT` in capitals, and
`"quotes"` for a phrase.

In [ ]:
# Errors in the last 24 hours, grouped by cause, each with what to do
lambdas.errors(FUNCTION)
# ↳ lambdas.errors(FUNCTION, since="7d")

In [ ]:
# The newest runs, a line each (status, start, run time, memory, cold start, what failed), opening to their lines
lambdas.logs(FUNCTION)

In [ ]:
# Search the logs: each matching run comes back whole, the matches marked
lambdas.logs(FUNCTION, search="KeyError", since="24h")
# ↳ lambdas.logs(FUNCTION, search="timeout OR AccessDenied", since="7d")
# ↳ lambdas.logs(FUNCTION, search='"Task timed out" NOT retry')

In [ ]:
# One run's lines, by its request ID (copy one from logs() or errors())
REQUEST_ID = "paste-a-request-id-here"
lambdas.logs(FUNCTION, request_id=REQUEST_ID)

In [ ]:
# Runs as rows you can filter, sort and group, with every field your JSON logs carry
lambdas.runs(FUNCTION, where="duration > 5s or error ~ KeyError")
# ↳ lambdas.runs(FUNCTION, where="cold and memory_share > 80%", sort="-duration")
# ↳ lambdas.runs(FUNCTION, group_by="status")

In [ ]:
# Run times, memory used, cold starts, and the memory it needs
lambdas.performance(FUNCTION)

## 📦 Code

In [ ]:
# The files in its deployment package, findings (secrets, size), and the handler's source
lambdas.code(FUNCTION)
# ↳ lambdas.code(FUNCTION, "app/handler.py")   # one file's source

In [ ]:
# What changed since its last published version: files, lines and settings
lambdas.diff(FUNCTION)

In [ ]:
# How a Python function's code works: a graph and the steps from the handler (nothing is run)
lambdas.how_it_works(FUNCTION)
# ↳ lambdas.how_it_works_page(FUNCTION)       # the same as a page to open in its own tab

In [ ]:
# Save its .zip package in lambda-downloads/ next to the notebook
lambdas.download(FUNCTION)

---
<a id="stepfunctions"></a>
# 🔀 AWS Step Functions

**Module:** `aws_analyzer.stepfunctions` &nbsp;·&nbsp; 📖 [Full guide](https://utkarsh5026.github.io/aws-analyzer/stepfunctions.html) &nbsp;·&nbsp; ⬆ [Back to contents](#contents)

| If you want to know… | Run |
| :-- | :-- |
| Which state machines fail, what do they cost? | `sfn.overview()` |
| Where and why do executions fail? | `sfn.failures(STATE_MACHINE)` |
| What happened in this execution, step by step? | `sfn.execution(EXECUTION)` |
| What does it look like? | `sfn.graph(STATE_MACHINE)` |

**Uses from the settings cell:** `STATE_MACHINE`, `EXECUTION`

In [ ]:
from aws_analyzer import StepFunctionsAnalyzer, StepFunctionsView

sfn = StepFunctionsView(StepFunctionsAnalyzer(**WHERE))
sfn.help()

## 🖱️ Explore by clicking

State machines, their executions with a waterfall and the Lambda lines behind each step, failures and the graph.

In [ ]:
sfn.explore()
# ↳ sfn.explore(STATE_MACHINE, tab="executions")
# ↳ sfn.explore(execution=EXECUTION)

## 🔀 State machines

In [ ]:
# Every state machine: executions and failures over 7 days, average run, cost, warnings
sfn.overview()

In [ ]:
# One state machine: type, executions, failures, cost, and what its definition risks
sfn.state_machine(STATE_MACHINE)

In [ ]:
# The flow drawn as a graph; with an execution, the path it took
sfn.graph(STATE_MACHINE)
# ↳ sfn.graph(STATE_MACHINE, execution=EXECUTION)
# ↳ sfn.graph(STATE_MACHINE, failures="7d")      # shade where executions stopped

<a id="sfn-executions"></a>
## 🏃 Executions

In [ ]:
# The latest executions: when each started, how it ended, how long it took
sfn.executions(STATE_MACHINE)
# ↳ sfn.executions(STATE_MACHINE, status="failed", since="30d")

In [ ]:
# One execution step by step on a time axis, with the Lambda log lines that broke it
sfn.execution(EXECUTION)

In [ ]:
# An execution beside the newest one that succeeded before it
sfn.compare(EXECUTION)

In [ ]:
# Why executions fail, grouped by the state they stopped in, with the Lambda run to read
sfn.failures(STATE_MACHINE)

---
<a id="apigateway"></a>
# 🚪 Amazon API Gateway

**Module:** `aws_analyzer.apigateway` &nbsp;·&nbsp; 📖 [Full guide](https://utkarsh5026.github.io/aws-analyzer/apigateway.html) &nbsp;·&nbsp; ⬆ [Back to contents](#contents)

| If you want to know… | Run |
| :-- | :-- |
| Which APIs get errors, who can call them, what do they cost? | `apigw.apis()` |
| Which calls fail, and why? | `apigw.errors(API)` |
| Show me the calls from the access log | `apigw.calls(API)` |
| Who can call each route? | `apigw.routes(API)` |

**Uses from the settings cell:** `API`

In [ ]:
from aws_analyzer import APIGatewayAnalyzer, APIGatewayView

apigw = APIGatewayView(APIGatewayAnalyzer(**WHERE))
apigw.help()

## 🖱️ Explore by clicking

APIs, their routes as a tree, calls from the access log (click one for the Lambda run behind it), performance and stages.

In [ ]:
apigw.explore()
# ↳ apigw.explore(API, tab="calls")

## 🚪 APIs

In [ ]:
# Every REST, HTTP and WebSocket API: requests, 4xx/5xx share, latency, who can call it, cost
apigw.apis()

In [ ]:
# One API: requests, errors, latency and cost, what needs attention, stages and routes
apigw.api_info(API)

In [ ]:
# Every route: who can call it, and where it goes; with curl commands to try it
apigw.routes(API)

In [ ]:
# Every stage: URL, throttling, logs, tracing, cache and WAF
apigw.stages(API)

<a id="apigw-calls"></a>
## 📞 Calls

In [ ]:
# The calls a stage's access log recorded, newest first
apigw.calls(API)
# ↳ apigw.calls(API, status="5xx", since="7d")
# ↳ apigw.calls(API, search="/orders")

In [ ]:
# What's failing: 4xx and 5xx by status, error and route, with the Lambda run behind each
apigw.errors(API)

In [ ]:
# How long calls take: typical and slowest, per route
apigw.performance(API)

---
<a id="athena"></a>
# 🏛️ Amazon Athena & the Glue Data Catalog

**Module:** `aws_analyzer.athena` &nbsp;·&nbsp; 📖 [Full guide](https://utkarsh5026.github.io/aws-analyzer/athena.html) &nbsp;·&nbsp; ⬆ [Back to contents](#contents)

| If you want to know… | Run |
| :-- | :-- |
| What do my queries cost, and which fail? | `athena.overview()` |
| Which queries cost the most? | `athena.expensive()` |
| Why do queries fail? | `athena.failures()` |
| Which tables point at empty folders? | `athena.catalog()` |

**Uses from the settings cell:** `ATHENA_TABLE`, `QUERY_ID`

In [ ]:
from aws_analyzer import AthenaAnalyzer, AthenaView

athena = AthenaView(AthenaAnalyzer(**WHERE))
athena.help()

## 🖱️ Explore by clicking

In [ ]:
athena.explore()
# ↳ athena.explore(tab="costs", since="30d")

<a id="athena-queries"></a>
## 🧾 Past queries

In [ ]:
# At a glance: what the last 7 days of queries scanned and cost, failures, workgroups, tables and crawlers
athena.overview()

In [ ]:
# Every query since a time, newest first: how it ended, what it scanned and cost
athena.queries("7d")
# ↳ athena.queries("24h", status="FAILED", workgroup="primary")

In [ ]:
# The costliest queries (the same query's runs added up), and how to make them cheaper
athena.expensive("30d")

In [ ]:
# Why queries failed, grouped by kind, each with what it means and the fix
athena.failures("7d")

In [ ]:
# One past query: its SQL, why it failed, what it cost, and the first rows of its results
athena.query(QUERY_ID)

## 📚 Data Catalog

In [ ]:
# Every database and table, and the tables whose S3 folder is empty or gone
athena.catalog()

In [ ]:
# One table: format, partitions, size in S3, columns, and the queries that read it
athena.table_info(ATHENA_TABLE)

In [ ]:
# Every Glue crawler: what it crawls, how often, how its last run went
athena.crawlers()

## ▶️ Run a query yourself

aws-analyzer never runs a query (Athena writes every result to S3). This shows the code to do it, and what it could cost.

In [ ]:
athena.query_code(f"SELECT * FROM {ATHENA_TABLE} LIMIT 10")

---
<a id="messaging"></a>
# 📨 SQS, SNS & EventBridge

**Module:** `aws_analyzer.messaging` &nbsp;·&nbsp; 📖 [Full guide](https://utkarsh5026.github.io/aws-analyzer/messaging.html) &nbsp;·&nbsp; ⬆ [Back to contents](#contents)

| If you want to know… | Run |
| :-- | :-- |
| What's waiting in my queues, what failed? | `msg.overview()` |
| Where does a message go from here, and where does it get lost? | `msg.follow(TOPIC)` |
| Which messages failed for good? | `msg.dead_letters()` |

**Uses from the settings cell:** `QUEUE`, `TOPIC`, `RULE`

In [ ]:
from aws_analyzer import MessagingAnalyzer, MessagingView

msg = MessagingView(MessagingAnalyzer(**WHERE))
msg.help()

## 🖱️ Explore by clicking

Every queue, topic and rule, a map of where messages go, dead letters, and a **Live** tab that watches a queue drain.

In [ ]:
msg.explore()
# ↳ msg.explore(QUEUE, tab="live")

## 🗺️ Where messages go

In [ ]:
# Every queue, topic and rule: messages waiting, the oldest one's age, dead letters, failed deliveries
msg.overview()

In [ ]:
# Where a message goes from a topic, queue, rule or bus, hop by hop, and where it gets lost
msg.follow(TOPIC)

In [ ]:
# Every dead-letter queue, the fullest first, and how to move the messages back
msg.dead_letters()

## 🔎 One queue, topic or rule

In [ ]:
msg.queue_info(QUEUE)

In [ ]:
msg.topic_info(TOPIC)

In [ ]:
msg.rule_info(RULE)

---
<a id="agent"></a>
# ✦ The agent — ask in plain English

**Module:** `aws_analyzer.agent` &nbsp;·&nbsp; 📖 [Full guide](https://utkarsh5026.github.io/aws-analyzer/agent.html) &nbsp;·&nbsp; ⬆ [Back to contents](#contents)

| If you want to know… | Run |
| :-- | :-- |
| Set up the model it answers with | `agent.setup()` |
| Ask a question and get the answer with what it checked | `agent.ask('...')` |
| Watch it work, step by step | `agent.chat()` |
| What can it read? | `agent.tools()` |

**Uses from the settings cell:** nothing — just your question. It needs a model: Amazon Bedrock, OpenAI, Azure OpenAI or any LangChain provider.

In [ ]:
from aws_analyzer import AgentView

agent = AgentView(**WHERE)                  # or AgentView(provider="bedrock", model="sonnet", **WHERE)
agent.help()

> 🔒 The agent only **reads**: every AWS call it makes is checked against a read-only list before it's sent.
> Big reads (many days of logs, a deep S3 listing) stop and ask first — answer with `agent.go()`, `agent.smaller()` or
> `agent.skip()`.

## 🔑 Set up

In [ ]:
# The steps to set up a model, as cells to copy, checked against this notebook
agent.setup()
# ↳ agent.setup("bedrock")       # or "openai", "azure"

In [ ]:
# Switch the model it answers with
agent.use_model("bedrock", "sonnet")
# ↳ agent.use_model("openai", "gpt-4.1")

## 💬 Ask

In [ ]:
# 🖱️ A chat window: type a question and watch each tool call as it runs
agent.chat()

In [ ]:
agent.ask("Which Lambda failed last night, and why?")

In [ ]:
# A follow-up: it remembers the conversation
agent.ask("Show me the log lines of the last failed run")

In [ ]:
agent.ask(f"Did the files uploaded to {BUCKET} last night get processed?")
# ↳ agent.new_conversation()       # start over

## ✋ Big reads

When an answer stops to ask before a big read, pick one:

In [ ]:
agent.go()          # make the big read
# ↳ agent.smaller()   # make the smaller read it suggested
# ↳ agent.skip()      # answer without it

## 🔎 Look back

In [ ]:
agent.transcript()               # every question and answer so far, with cost
# ↳ agent.save_charts()            # save the last answer's charts as pictures

## 🧰 What it can do

In [ ]:
agent.tools()                    # each tool, the AWS reads it makes, the model and the budget
# ↳ agent.runbooks()               # the step-by-step guides it follows

## 🗂 Saved chats, 🧠 memory and 🔌 MCP servers

These need the agent made with `chats=True` / `memory=True`.

In [ ]:
keeper = AgentView(chats=True, memory=True, **WHERE)
keeper.chats()                   # earlier chats, by title
# ↳ keeper.open_chat(1)            # carry one on
# ↳ keeper.rename_chat("Nightly ETL failures")
# ↳ keeper.delete_chat(3)

In [ ]:
keeper.memory()                  # what it remembers about this account
# ↳ keeper.forget("orders-etl")
# ↳ keeper.forget_all()

In [ ]:
agent.mcp()                      # your MCP servers and their tools
# ↳ agent.add_mcp("npx -y @modelcontextprotocol/server-memory")
# ↳ agent.import_mcp()             # servers from VS Code, Claude Desktop, Cursor ...
# ↳ agent.disable_mcp("memory"); agent.enable_mcp("memory"); agent.reconnect_mcp(); agent.remove_mcp("memory")

In [ ]:
# ✦ Ask button in every explorer window, asking the agent about what that window shows
from aws_analyzer import set_assistant

set_assistant(agent)

---
<a id="housekeeping"></a>
# 🧹 Housekeeping

What aws-analyzer saved on this machine (S3 and Lambda downloads, the agent's charts and sandbox cache), and how to
remove it. It only ever removes what it made itself.  ⬆ [Back to contents](#contents)

In [ ]:
from aws_analyzer import clean_up, local_files

local_files()                    # what's there and how big

In [ ]:
clean_up(dry_run=True)           # what would be removed; drop dry_run=True to remove it
# ↳ clean_up("s3")                 # only the S3 downloads ("s3", "lambda", "sandbox", "charts" or "all")

---
<a id="more"></a>
# 🎛️ Good to know

| To… | Do this |
| :-- | :-- |
| see times in your own zone | `set_timezone("Asia/Kolkata")` (or `AWS_ANALYZER_TZ=...` for good) |
| change how reports look | `set_theme("soft")` — `console`, `soft`, `editorial`, `brand`, `classic` |
| get plain text instead of HTML | `S3View(mode="text")` — every View takes it |
| show every row of long tables | `S3View(max_rows=0)` |
| turn off progress bars | `S3View(progress="off")` |
| get **data** instead of a report | every View has `.core`: `s3.core.summarize(FOLDER)` returns a dataclass, many have `.to_df()` |
| read another account or region | change `PROFILE` / `REGION` in [⚙️ Your settings](#settings) and re-run the service's first cell |

📖 Full documentation: **https://utkarsh5026.github.io/aws-analyzer/** &nbsp;·&nbsp; ⬆ [Back to contents](#contents)